# Multi-product validation of SWOT water surface elevation in the Pantanal Rivers

This single-notebook workflow is linked to a manuscript currently under submission and not yet published:

**Multi-product validation of SWOT water surface elevation in the Pantanal Rivers**

Jahdy Moreno-Oliveira a,b*, Alice Fassoni-Andrade b, Mark A. Trigg c, Daniel Medeiros Moreira d, Evlyn Marcia Leão de Moraes Novoa

a National Institute for Space Research (INPE), São Jose dos Campos, Brazil  
b Institute of Geosciences, University of Brasília (UnB), Brasília, Brazil  
c School of Civil Engineering, University of Leeds, Leeds, United Kingdom  
d Geological Survey of Brazil, Rio de Janeiro, Brazil

Manuscript status: under submission; not yet published.

This notebook is intentionally self-contained. It does not require a companion `.py` file. It uses only Python standard-library modules for reading tables, matching time series, filtering outliers, and calculating metrics. If `matplotlib` is already available, it also exports figures; if not, table outputs are still generated.


In [ ]:
# Optional: uncomment only if your environment does not already have plotting support.
# The notebook generates the CSV results without installing extra packages.
# %pip install matplotlib


## Input Contract

Run this notebook from the same working directory used by the processing workflow. The default paths below are intentionally relative to that directory.

```text
<working-directory>/
├── analises/
│   └── tabelas/
│       ├── ana_metadados.csv
│       └── correspondencia_altimetria_convencional.csv
├── dados_processados/
│   └── ana/
│       ├── sem_nivelamento/{station_id}.csv
│       └── nivelamento/{station_id}_nivelada.csv
├── analise_final/
│   ├── water_mask_15m.shp
│   └── Estacao_{station_id}/
│       ├── pixc_complete_raw.csv
│       ├── raster_complete_raw.csv
│       ├── node_complete_raw.csv
│       ├── reach_complete_raw.csv
│       └── alt_conv_{station_id}.csv
├── analysis_absolute/
├── analysis_temporal/
├── analysis_drivers/
└── figures/
    ├── main/
    └── supplementary/
```

Minimum required columns:

- Station metadata: a station-code column named `station`, `codigo`, `cod_ana`, `codigo_ana`, or `id`.
- In situ CSVs: a date column plus one water-level column such as `wse`, `wse_ana`, `wse_egm08`, `Cota`, `cota`, `nivel`, or `stage`.
- SWOT product CSVs: a date/time column plus WSE columns matching each product (`wse_media` for PIXC/Raster; `wse` for Node/Reach).
- Altimetry CSVs: a date column plus `wse`.


In [ ]:
from pathlib import Path

BASE_PATH = Path.cwd()

CFG = {
    "base_path": BASE_PATH,
    "date_start": "2023-03-31",
    "date_end": "2025-08-01",
    "calval_cutoff_date": "2023-07-01",
    "match_window_days": 3,
    "station_filter": [],
    "metadata_candidates": [
        Path("analises/tabelas/ana_metadados.csv"),
        Path("ana_metadados.csv"),
    ],
    "in_situ_unleveled_dir": Path("dados_processados/ana/sem_nivelamento"),
    "in_situ_gnss_leveled_dir": Path("dados_processados/ana/nivelamento"),
    "station_dir_template": "analise_final/Estacao_{station}",
    "absolute_dir": Path("analysis_absolute"),
    "temporal_dir": Path("analysis_temporal"),
    "drivers_dir": Path("analysis_drivers"),
    "figures_main_dir": Path("figures/main"),
    "figures_supplementary_dir": Path("figures/supplementary"),
}

CFG


In [ ]:

import csv
import json
import math
import re
from collections import defaultdict
from datetime import datetime, date, timedelta
from pathlib import Path
from statistics import median

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

PRODUCTS_SWOT = ["pixc", "raster", "node", "reach"]
PRODUCTS_ALL = PRODUCTS_SWOT + ["altimetry"]
PRODUCT_LABELS = {"pixc": "PIXC", "raster": "Raster", "node": "Node", "reach": "Reach", "altimetry": "Altimetry"}
PRODUCT_COLORS = {"pixc": "#e67e22", "raster": "#2e7d32", "node": "#c0392b", "reach": "#6c5ce7", "altimetry": "#6f6f6f"}

SWOT_SCHEMAS = {
    "pixc": {"file": "pixc_complete_raw.csv", "date_cols": ["data", "arquivo"], "wse_cols": ["wse_media", "wse", "height_media"]},
    "raster": {"file": "raster_complete_raw.csv", "date_cols": ["data", "arquivo"], "wse_cols": ["wse_media", "wse", "height"]},
    "node": {"file": "node_complete_raw.csv", "date_cols": ["data", "time_str", "timestamp_arquivo", "arquivo_origem"], "wse_cols": ["wse", "WSE", "wse_sm"]},
    "reach": {"file": "reach_complete_raw.csv", "date_cols": ["data", "time_str", "timestamp_arquivo", "arquivo_origem"], "wse_cols": ["wse", "WSE"]},
}


def pjoin(*parts):
    return Path(CFG["base_path"]).joinpath(*parts)


def cfg_path(key):
    return pjoin(CFG[key])


def ensure_dirs():
    for key in ["absolute_dir", "temporal_dir", "drivers_dir", "figures_main_dir", "figures_supplementary_dir"]:
        cfg_path(key).mkdir(parents=True, exist_ok=True)


def read_text_sample(path, size=4096):
    with open(path, "r", encoding="utf-8-sig", errors="replace") as f:
        return f.read(size)


def sniff_dialect(path):
    sample = read_text_sample(path)
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t")
    except Exception:
        dialect = csv.excel
        dialect.delimiter = ","
        return dialect


def read_table(path):
    path = Path(path)
    if not path.exists():
        return []
    dialect = sniff_dialect(path)
    with open(path, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
        reader = csv.DictReader(f, dialect=dialect)
        return [{(k or "").strip(): (v or "").strip() for k, v in row.items()} for row in reader]


def write_table(path, rows, fieldnames=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if fieldnames is None:
        keys = []
        seen = set()
        for row in rows:
            for key in row.keys():
                if key not in seen:
                    seen.add(key)
                    keys.append(key)
        fieldnames = keys
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, "") for key in fieldnames})


def normalize_station_id(value):
    text = str(value or "").replace("\ufeff", "").replace("\xa0", " ").strip()
    text = re.sub(r"\s+", "", text)
    if re.fullmatch(r"\d+\.0+", text):
        return text.split(".")[0]
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except Exception:
        pass
    return text


def to_float(value):
    if value is None:
        return None
    text = str(value).strip().replace("%", "").replace(" ", "")
    if text == "" or text.lower() in {"nan", "none", "null", "na"}:
        return None
    candidates = [text, text.replace(",", "."), text.replace(".", "").replace(",", ".")]
    for candidate in candidates:
        try:
            out = float(candidate)
            if math.isfinite(out):
                return out
        except Exception:
            pass
    return None


def parse_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text:
        return None
    patterns = [
        "%Y-%m-%d", "%Y-%m-%d %H:%M:%S", "%Y/%m/%d", "%d/%m/%Y", "%d-%m-%Y",
        "%Y%m%dT%H%M%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M:%SZ",
    ]
    match = re.search(r"(\d{8}T\d{6})", text)
    if match:
        text = match.group(1)
    for pattern in patterns:
        try:
            return datetime.strptime(text[:len(datetime.now().strftime(pattern))] if "%" in pattern else text, pattern)
        except Exception:
            pass
    try:
        number = float(text)
        if number > 1e12:
            return datetime.fromtimestamp(number / 1e9)
        if number > 1e9:
            return datetime.fromtimestamp(number)
    except Exception:
        pass
    return None


def date_key(dt):
    return dt.strftime("%Y-%m-%d") if isinstance(dt, datetime) else ""


def between_dates(dt):
    if not dt:
        return False
    return datetime.fromisoformat(CFG["date_start"]) <= dt <= datetime.fromisoformat(CFG["date_end"])


def orbital_phase(dt):
    if not dt:
        return ""
    return "CalVal" if dt < datetime.fromisoformat(CFG["calval_cutoff_date"]) else "Nominal"


def pick(row, candidates):
    for col in candidates:
        if col in row and str(row[col]).strip() != "":
            return row[col], col
    return "", ""


def ranks(values):
    indexed = sorted((value, idx) for idx, value in enumerate(values))
    out = [0.0] * len(values)
    i = 0
    while i < len(indexed):
        j = i
        while j + 1 < len(indexed) and indexed[j + 1][0] == indexed[i][0]:
            j += 1
        rank = (i + j + 2) / 2.0
        for _, idx in indexed[i:j + 1]:
            out[idx] = rank
        i = j + 1
    return out


def pearson(x, y):
    n = len(x)
    if n < 2:
        return None
    mx = sum(x) / n
    my = sum(y) / n
    sx = sum((v - mx) ** 2 for v in x)
    sy = sum((v - my) ** 2 for v in y)
    if sx == 0 or sy == 0:
        return None
    return sum((a - mx) * (b - my) for a, b in zip(x, y)) / math.sqrt(sx * sy)


def spearman(x, y):
    clean = [(a, b) for a, b in zip(x, y) if a is not None and b is not None]
    if len(clean) < 2:
        return None
    rx = ranks([a for a, _ in clean])
    ry = ranks([b for _, b in clean])
    return pearson(rx, ry)


def calc_metrics(rows, ref="wse_insitu", sat="wse_sat", include_bias=True):
    pairs = [(to_float(r.get(ref)), to_float(r.get(sat))) for r in rows]
    pairs = [(a, b) for a, b in pairs if a is not None and b is not None]
    out = {"n_obs": len(pairs), "mae": "", "rmse": "", "spearman_r": "", "bias": "" if include_bias else ""}
    if len(pairs) < 2:
        return out
    errors = [b - a for a, b in pairs]
    out["mae"] = sum(abs(e) for e in errors) / len(errors)
    out["rmse"] = math.sqrt(sum(e * e for e in errors) / len(errors))
    out["spearman_r"] = spearman([a for a, _ in pairs], [b for _, b in pairs])
    if include_bias:
        out["bias"] = sum(errors) / len(errors)
    return out


def hampel_filter(rows, value_col="wse_sat", window=5, n_sigma=3.0):
    rows = sorted(rows, key=lambda r: r.get("data") or datetime.min)
    audit = []
    keep = []
    half = window // 2
    values = [to_float(r.get(value_col)) for r in rows]
    for i, row in enumerate(rows):
        local = [v for v in values[max(0, i - half): i + half + 1] if v is not None]
        value = values[i]
        flag = False
        med = mad = threshold = deviation = ""
        if value is not None and len(local) >= 3:
            med = median(local)
            deviations = [abs(v - med) for v in local]
            mad = median(deviations)
            threshold = n_sigma * 1.4826 * mad
            deviation = abs(value - med)
            flag = threshold > 0 and deviation > threshold
        audited = dict(row)
        audited.update({
            "hampel_median": med,
            "hampel_mad": mad,
            "hampel_threshold": threshold,
            "hampel_abs_deviation": deviation,
            "is_hampel_outlier": flag,
            "is_outlier_any": flag,
        })
        audit.append(audited)
        if not flag:
            keep.append(dict(row))
    return keep, audit


def infer_hydro_phase(values):
    numeric = [v for v in values if v is not None]
    if len(numeric) < 4:
        return [""] * len(values)
    sorted_vals = sorted(numeric)
    q25 = sorted_vals[int(0.25 * (len(sorted_vals) - 1))]
    q75 = sorted_vals[int(0.75 * (len(sorted_vals) - 1))]
    phases = []
    for i, value in enumerate(values):
        if value is None:
            phases.append("")
        elif value >= q75:
            phases.append("High")
        elif value <= q25:
            phases.append("Low")
        else:
            prev = values[i - 1] if i > 0 and values[i - 1] is not None else value
            phases.append("Rising" if value >= prev else "Falling")
    return phases


def load_metadata():
    candidates = [pjoin(p) for p in CFG["metadata_candidates"]]
    path = None
    for candidate in candidates:
        if candidate.exists():
            path = candidate
            break
    if path is None:
        checked = ", ".join(str(p) for p in candidates)
        raise FileNotFoundError(f"Station metadata not found. Checked: {checked}")
    rows = read_table(path)
    out = []
    for row in rows:
        station, _ = pick(row, ["station", "codigo", "cod_ana", "codigo_ana", "id"])
        station = normalize_station_id(station)
        if not station:
            continue
        if CFG["station_filter"] and station not in set(map(str, CFG["station_filter"])):
            continue
        out.append({
            "station": station,
            "station_name": pick(row, ["station_name", "nome", "name"])[0],
            "river": pick(row, ["river", "rio", "river_name"])[0],
            "latitude": pick(row, ["latitude", "lat"])[0],
            "longitude": pick(row, ["longitude", "lon", "long"])[0],
            "width": pick(row, ["width", "largura"])[0],
            "drainage_area": pick(row, ["drainage_area", "area_drenagem"])[0],
            "subregion": pick(row, ["subregion", "subregiao"])[0],
            "pantanal": pick(row, ["pantanal"])[0],
        })
    return out


def load_insitu(station, leveled=False):
    base = cfg_path("in_situ_gnss_leveled_dir" if leveled else "in_situ_unleveled_dir")
    if leveled:
        files = [base / f"{station}_nivelada.csv", base / f"{station}.csv", base / f"{station}_leveled.csv"]
        files.extend(sorted(base.glob(f"{station}*nivel*.csv")))
    else:
        files = [base / f"{station}.csv"]
    path = next((p for p in files if p.exists()), None)
    if path is None:
        return []
    raw = read_table(path)
    out = []
    wse_values = []
    for row in raw:
        date_value, _ = pick(row, ["data", "date", "datetime", "data_ref", "time"])
        wse_value, _ = pick(row, ["wse_swot_ref", "wse_ana", "wse_egm08", "wse", "WSE", "Cota", "cota", "nivel", "stage"])
        dt = parse_date(date_value)
        wse = to_float(wse_value)
        if not dt or wse is None or not between_dates(dt):
            continue
        phase, _ = pick(row, ["hydro_phase", "phase", "fase", "fase_hidrologica"])
        out.append({"station": station, "data": dt, "date": date_key(dt), "wse_insitu": wse, "hydro_phase": phase})
        wse_values.append(wse)
    inferred = infer_hydro_phase(wse_values)
    for row, phase in zip(out, inferred):
        if not row["hydro_phase"]:
            row["hydro_phase"] = phase
    return sorted(out, key=lambda r: r["data"])


def station_dir(template, station):
    return pjoin(template.format(station=station))


def load_swot(station, product):
    schema = SWOT_SCHEMAS[product]
    base = station_dir(CFG["station_dir_template"], station)
    candidates = [base / schema["file"]]
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        return []
    rows = []
    for row in read_table(path):
        date_value, _ = pick(row, schema["date_cols"])
        wse_value, wse_col = pick(row, schema["wse_cols"])
        dt = parse_date(date_value)
        wse = to_float(wse_value)
        if not dt or wse is None or not between_dates(dt) or wse <= -1e9:
            continue
        out = dict(row)
        out.update({"station": station, "product": product, "data": dt, "date": date_key(dt), "wse_sat": wse, "orbital_phase": orbital_phase(dt), "wse_source_column": wse_col})
        rows.append(out)
    return sorted(rows, key=lambda r: r["data"])


def load_altimetry(station):
    base = station_dir(CFG["station_dir_template"], station)
    files = list(base.glob("alt_conv_*.csv"))
    grouped = defaultdict(list)
    for path in files:
        for row in read_table(path):
            date_value, _ = pick(row, ["date", "data", "datetime"])
            wse_value, _ = pick(row, ["wse", "WSE", "water_level", "height"])
            dt = parse_date(date_value)
            wse = to_float(wse_value)
            if dt and wse is not None and between_dates(dt):
                grouped[date_key(dt)].append(wse)
    out = []
    for d, values in grouped.items():
        dt = datetime.fromisoformat(d)
        out.append({"station": station, "product": "altimetry", "data": dt, "date": d, "wse_sat": sum(values) / len(values), "orbital_phase": orbital_phase(dt)})
    return sorted(out, key=lambda r: r["data"])


def quality_filter(rows, product):
    out = []
    for row in rows:
        if product == "node" and to_float(row.get("node_q")) is not None and to_float(row.get("node_q")) >= 2:
            continue
        if product == "reach" and to_float(row.get("reach_q")) is not None and to_float(row.get("reach_q")) >= 2:
            continue
        if product == "raster" and to_float(row.get("wse_qual_media")) is not None and to_float(row.get("wse_qual_media")) >= 1.8:
            continue
        out.append(dict(row))
    return out


def nearest_match_no_reuse(ref_rows, sat_rows):
    ref_rows = sorted(ref_rows, key=lambda r: r["data"])
    sat_rows = sorted(sat_rows, key=lambda r: r["data"])
    used = set()
    window = timedelta(days=CFG["match_window_days"])
    matched = []
    for ref in ref_rows:
        candidates = []
        for idx, sat in enumerate(sat_rows):
            if idx in used:
                continue
            delta = abs(sat["data"] - ref["data"])
            if delta <= window:
                candidates.append((delta, idx, sat))
        if not candidates:
            continue
        _, idx, sat = min(candidates, key=lambda item: item[0])
        used.add(idx)
        row = dict(sat)
        row.update({
            "date_ref": date_key(ref["data"]),
            "date_sat": date_key(sat["data"]),
            "wse_insitu": ref["wse_insitu"],
            "wse_sat": sat["wse_sat"],
            "hydro_phase": ref.get("hydro_phase", ""),
            "orbital_phase": sat.get("orbital_phase", ""),
        })
        row.pop("data", None)
        matched.append(row)
    return matched


def available_leveled_stations():
    base = cfg_path("in_situ_gnss_leveled_dir")
    stations = set()
    if base.exists():
        for path in base.glob("*.csv"):
            stem = path.stem
            for suffix in ["_nivelada", "_leveled"]:
                if stem.endswith(suffix):
                    stem = stem[: -len(suffix)]
            stations.add(normalize_station_id(stem))
    stations.discard("")
    return stations


def add_matches(stages, audit, meta, station, product, ref_rows, station_group):
    sat = load_altimetry(station) if product == "altimetry" else load_swot(station, product)
    if not sat:
        return
    stage_tables = {"raw": sat, "quality_filtered": quality_filter(sat, product)}
    stage_tables["outlier_filtered"], audit_rows = hampel_filter(stage_tables["quality_filtered"])
    audit.extend([dict(r, station=station, product=product, station_group=station_group) for r in audit_rows])
    for stage, rows in stage_tables.items():
        matches = nearest_match_no_reuse(ref_rows, rows)
        for match in matches:
            error = to_float(match["wse_sat"]) - to_float(match["wse_insitu"])
            match.update(meta)
            match.update({
                "station": station,
                "product": product,
                "station_group": station_group,
                "lineage_stage": stage,
                "signed_error": error,
                "error": error,
                "abs_error": abs(error),
            })
        stages[stage].extend(matches)


def build_matchups(metadata):
    leveled = available_leveled_stations()
    stages = {"raw": [], "quality_filtered": [], "outlier_filtered": []}
    audit = []
    for meta in metadata:
        station = meta["station"]
        refs = {
            "unleveled": load_insitu(station, leveled=False),
            "leveled": load_insitu(station, leveled=True) if station in leveled else [],
        }
        for station_group, ref_rows in refs.items():
            if not ref_rows:
                continue
            for product in PRODUCTS_ALL:
                add_matches(stages, audit, meta, station, product, ref_rows, station_group)
    return stages, audit

def group_by(rows, keys):
    groups = defaultdict(list)
    for row in rows:
        groups[tuple(row.get(k, "") for k in keys)].append(row)
    return groups


def aggregate_metrics(rows, keys):
    out = []
    for values, group in group_by(rows, keys).items():
        rec = {k: v for k, v in zip(keys, values)}
        rec.update(calc_metrics(group))
        rec["n_stations"] = len({r.get("station") for r in group})
        out.append(rec)
    return out


def dynamic_metrics(rows):
    station_rows = []
    normalized = []
    for (station, product), group in group_by(rows, ["station", "product"]).items():
        if len(group) < 5:
            continue
        ref_vals = [to_float(r.get("wse_insitu")) for r in group]
        sat_vals = [to_float(r.get("wse_sat")) for r in group]
        if any(v is None for v in ref_vals + sat_vals):
            continue
        if max(ref_vals) == min(ref_vals) or max(sat_vals) == min(sat_vals):
            continue
        ref_norm = [(v - min(ref_vals)) / (max(ref_vals) - min(ref_vals)) for v in ref_vals]
        sat_norm = [(v - min(sat_vals)) / (max(sat_vals) - min(sat_vals)) for v in sat_vals]
        rho = spearman(ref_norm, sat_norm)
        station_rows.append({"station": station, "product": product, "n_obs": len(group), "spearman_r": rho, "station_group": group[0].get("station_group", "")})
        for row, rn, sn in zip(group, ref_norm, sat_norm):
            normalized.append({"station": station, "product": product, "date_ref": row.get("date_ref"), "insitu_norm": rn, "sat_norm": sn, "station_group": row.get("station_group", ""), "hydro_phase": row.get("hydro_phase", "")})
    overall = []
    for (product,), group in group_by(station_rows, ["product"]).items():
        rhos = [to_float(r.get("spearman_r")) for r in group if to_float(r.get("spearman_r")) is not None]
        overall.append({"product": product, "n_station_series": len(group), "n_obs": sum(int(r.get("n_obs", 0)) for r in group), "median_spearman": median(rhos) if rhos else "", "mean_spearman": sum(rhos) / len(rhos) if rhos else ""})
    return station_rows, overall, normalized


def savefig(name):
    if plt is None:
        return None
    path = cfg_path("figures_main_dir") / f"{name}.png"
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(path, bbox_inches="tight", dpi=300)
    plt.close()
    return str(path)


def make_figures(tables):
    manifest = []
    if plt is None:
        return manifest
    metrics = tables.get("metrics_absolute_overall", [])
    if metrics:
        for metric in ["mae", "rmse", "bias"]:
            plt.figure(figsize=(6, 4))
            xs = [PRODUCT_LABELS.get(r["product"], r["product"]) for r in metrics]
            ys = [to_float(r.get(metric)) or 0 for r in metrics]
            colors = [PRODUCT_COLORS.get(r["product"], "#777") for r in metrics]
            plt.bar(xs, ys, color=colors)
            plt.ylabel(metric.upper() if metric != "bias" else "Bias (m)")
            plt.grid(axis="y", alpha=0.25)
            out = savefig(f"fig_absolute_{metric}")
            if out:
                manifest.append({"figure_path": out})
    dyn = tables.get("metrics_dynamic_overall", [])
    if dyn:
        plt.figure(figsize=(7, 4))
        xs = [PRODUCT_LABELS.get(r["product"], r["product"]) for r in dyn]
        ys = [to_float(r.get("median_spearman")) or 0 for r in dyn]
        colors = [PRODUCT_COLORS.get(r["product"], "#777") for r in dyn]
        plt.bar(xs, ys, color=colors)
        plt.ylim(-1, 1)
        plt.ylabel("Median Spearman")
        plt.grid(axis="y", alpha=0.25)
        out = savefig("fig_dynamic_spearman_summary")
        if out:
            manifest.append({"figure_path": out})
    return manifest


def split_by_station_group(stages, station_group):
    return {stage: [r for r in rows if r.get("station_group") == station_group] for stage, rows in stages.items()}


def write_stage_tables(out_dir, stages):
    for stage, rows in stages.items():
        write_table(out_dir / f"matchups_{stage}.csv", rows)


def write_named_tables(out_dir, tables, output_names):
    for key, filenames in output_names.items():
        for filename in filenames:
            write_table(out_dir / filename, tables.get(key, []))


def run_pipeline():
    ensure_dirs()
    metadata = load_metadata()
    stages, audit = build_matchups(metadata)
    absolute_stages = split_by_station_group(stages, "leveled")
    temporal_stages = split_by_station_group(stages, "unleveled")

    out_abs = cfg_path("absolute_dir")
    out_tmp = cfg_path("temporal_dir")
    out_drv = cfg_path("drivers_dir")

    tables = {}
    for stage, rows in stages.items():
        tables[f"matchups_{stage}"] = rows
    write_stage_tables(out_abs, absolute_stages)
    write_stage_tables(out_tmp, temporal_stages)
    write_table(out_abs / "outlier_filter_audit.csv", [r for r in audit if r.get("station_group") == "leveled"])
    write_table(out_tmp / "outlier_filter_audit.csv", [r for r in audit if r.get("station_group") == "unleveled"])

    absolute = absolute_stages["outlier_filtered"]
    temporal = temporal_stages["outlier_filtered"]
    tables["metrics_absolute_overall"] = aggregate_metrics(absolute, ["product"])
    tables["metrics_absolute_by_station"] = aggregate_metrics(absolute, ["station", "product"])
    tables["metrics_absolute_by_orbital_phase"] = aggregate_metrics(absolute, ["product", "orbital_phase"])
    tables["metrics_absolute_by_hydro_phase"] = aggregate_metrics([r for r in absolute if r.get("product") in PRODUCTS_SWOT], ["product", "hydro_phase"])
    tables["metrics_dynamic_by_station"], tables["metrics_dynamic_overall"], tables["normalized_series_for_examples"] = dynamic_metrics(temporal)

    absolute_outputs = {
        "metrics_absolute_overall": ["metrics_absolute_overall.csv", "metrics_overall.csv"],
        "metrics_absolute_by_station": ["metrics_absolute_by_station.csv", "metrics_by_station.csv"],
        "metrics_absolute_by_orbital_phase": ["metrics_absolute_by_orbital_phase.csv", "metrics_by_product_orbital_phase.csv"],
        "metrics_absolute_by_hydro_phase": ["metrics_absolute_by_hydro_phase.csv", "error_by_hydrological_phase.csv"],
    }
    temporal_outputs = {
        "metrics_dynamic_by_station": ["metrics_dynamic_by_station.csv"],
        "metrics_dynamic_overall": ["metrics_dynamic_overall.csv"],
        "normalized_series_for_examples": ["normalized_series_for_examples.csv"],
    }
    write_named_tables(out_abs, tables, absolute_outputs)
    write_named_tables(out_tmp, tables, temporal_outputs)

    driver_placeholder = [{"note": "Driver-analysis tables are not recalculated in this compact notebook."}]
    write_table(out_drv / "driver_analysis_note.csv", driver_placeholder)

    summary = {
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "base_path": str(CFG["base_path"]),
        "date_start": CFG["date_start"],
        "date_end": CFG["date_end"],
        "match_window_days": CFG["match_window_days"],
        "n_metadata_stations": len(metadata),
        "n_matchups_raw_all": len(stages["raw"]),
        "n_matchups_absolute_outlier_filtered": len(absolute),
        "n_matchups_temporal_outlier_filtered": len(temporal),
        "n_absolute_station_product_metrics": len(tables["metrics_absolute_by_station"]),
        "n_dynamic_station_product_series": len(tables["metrics_dynamic_by_station"]),
    }
    (out_abs / "execution_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    (out_tmp / "execution_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    tables["execution_summary"] = [summary]
    tables["figure_manifest"] = make_figures(tables)
    write_table(cfg_path("figures_main_dir") / "figure_manifest.csv", tables["figure_manifest"])
    return tables


In [ ]:
input_items = {
    "metadata_primary": pjoin(CFG["metadata_candidates"][0]),
    "metadata_secondary": pjoin(CFG["metadata_candidates"][1]),
    "in_situ_unleveled": cfg_path("in_situ_unleveled_dir"),
    "in_situ_gnss_leveled": cfg_path("in_situ_gnss_leveled_dir"),
    "station_product_root": pjoin(Path(CFG["station_dir_template"].split("/Estacao_")[0])),
}

for name, path in input_items.items():
    print(f"{name}: {path} | exists={path.exists()}")


In [ ]:

tables = run_pipeline()

for name in [
    "metrics_absolute_overall",
    "metrics_absolute_by_station",
    "metrics_dynamic_overall",
    "figure_manifest",
]:
    print(f"\n{name}")
    for row in tables.get(name, [])[:5]:
        print(row)


In [ ]:
summary_path = cfg_path("absolute_dir") / "execution_summary.json"
print(summary_path)
print(summary_path.read_text(encoding="utf-8"))


## Main Outputs

Absolute validation tables are written to `analysis_absolute/`:

- `matchups_raw.csv`
- `matchups_quality_filtered.csv`
- `matchups_outlier_filtered.csv`
- `metrics_overall.csv`
- `metrics_by_station.csv`
- `metrics_by_product_orbital_phase.csv`
- `error_by_hydrological_phase.csv`
- `outlier_filter_audit.csv`
- `execution_summary.json`

Dynamic validation tables are written to `analysis_temporal/`:

- `matchups_raw.csv`
- `matchups_quality_filtered.csv`
- `matchups_outlier_filtered.csv`
- `metrics_dynamic_by_station.csv`
- `metrics_dynamic_overall.csv`
- `normalized_series_for_examples.csv`
- `outlier_filter_audit.csv`
- `execution_summary.json`

Figures are written to `figures/main/` when `matplotlib` is available.
